# Aim
The purpose of this notebook is to demo AITune module wrapper.

To run this, install extras `demo`.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..
%pwd

In [ ]:
import copy
import os
from logging import basicConfig

import torch

from aitune.global_context import BATCH_SIZE_KEY, global_context
from aitune.torch.backend.torch_inductor_backend import TorchInductorBackend
from aitune.torch.module.wrapper_module import Module
from aitune.torch.tune_strategy.one_backend_strategy import OneBackendStrategy

log_level = os.environ.get("AITUNE_LOG_LEVEL", "INFO")
basicConfig(level=log_level, format="%(asctime)s - %(levelname)s - %(message)s", force=True)

## Simple identity model
Let's start with something simple.

In [ ]:
class Identity(torch.nn.Module):

    def __init__(self):
        super().__init__()

    def forward(self, x, **kwargs):
        return x

#### Let's wrap module

In [ ]:
model = Identity()
module = Module(model, "demo-identity")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Record some samples and check the graph specs

A `graph spec` has a `name` and corresponding 
- `input_spec` which defines input `args` and `kwargs` to the torch `module.forward function`.
- `output_spec` which defines the output of the function

In [ ]:
module(1, a=True)
module.graph_specs

Let's decipher the graph specs' `input_spec`. It consists of two parts:
- first are `args` passed to the function
- second are `kwargs`

In [ ]:
def print_graph_spec(module):
    for graph_spec in module.graph_specs:
        print("Graph", graph_spec.name)
        print("- input_spec:", graph_spec.input_spec.describe())  # noqa: T201
        print("- output_spec:", graph_spec.output_spec.describe())  # noqa: T201

In [ ]:
print_graph_spec(module)

Let's record another sample.

In [ ]:
module(2)
print_graph_spec(module)

Let's record a sample with a tensor.

In [ ]:
module(torch.randn(1, 1))
print_graph_spec(module)

As can be seen, third graph has tensor as input with shapes `[1, 1]` and min and max shapes being the same.

If we record more samples of different shapes, the graph will have dynamic dimensions calculated.

In [ ]:
module(torch.randn(2, 2))
print_graph_spec(module)

As can be seen, dynamic shapes are named `dim0`, `dim1` - they were calculated because we recorded same tensor with different shapes meaning they have to be dynamic.

The above also shows that graph has seen min and max shapes of `[1, 1]` and `[2, 2]` respectively i.e. first dimension is from 1 to 2, same for the second.

In [ ]:
module.state

#### Automatic batch dimension detection

As we have seen module detects dynamic axes just by looking at different input data. However for checking performance and sanity aitune needs to detect batch axis - which is special type of dynamic axis i.e. grows with a batch size. One example could be LLM pipeline where we have batch and sequence length dynamic axes `B` and `L` but only `B` grows with a batch size.

In order to detect batch dimension Module wrapper has to have information about current global batch size. This is because the `global batch size` can be different than `local batch size` e.g. SDXL model stacks input tensor and with `global bs=1` a module can receive `local bs=2`.

When you perform tuning, the aitune takes dataloader and populates information about `batch size` so that module wrapper can obtain it. Since we are demonstrating module wrapper in isolation so we have to provide that information ourselves.

In [ ]:
with global_context:
    global_context[BATCH_SIZE_KEY] = 1
    module(torch.randn(1, 2, 3))
    global_context[BATCH_SIZE_KEY] = 2
    module(torch.randn(2, 4, 3,))

In [ ]:
print_graph_spec(module)


As can be seen, the last graph has batch axis `batch0` which grows with batch size and `batch1` which grows twice the batch size.

#### Check changing state to passthrough and back to recording.
In passthrough samples are not recorded. Hence the graph should not be altered.

In [ ]:
module.enable_passthrough()
module(3)  # a new graph - but should not be detected
print_graph_spec(module)

#### Let's try tune dry-run

In [ ]:
module.enable_recording()
module.tune(dry_run=True, device=device)

In [ ]:
module.state

# Resnet50

In [ ]:
# example from https://huggingface.co/docs/timm/en/models/resnet
import urllib
from pathlib import Path

import timm
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform


In [ ]:
model = timm.create_model('resnet50', pretrained=True)
model.to('cuda')
model.eval()
config = resolve_data_config({}, model=model)
transform = create_transform(**config)

url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
if not Path(filename).exists():
    urllib.request.urlretrieve(url, filename)

img = Image.open(filename).convert('RGB')
data = transform(img).unsqueeze(0).to('cuda')  # transform and add batch dimension
data.shape

In [ ]:
with torch.no_grad():
    out = model(data)
ref_probs = torch.nn.functional.softmax(out[0], dim=0)


#### Let's wrap module

In [ ]:
module = Module(model, "demo-resnet50")

#### Record some samples

In [ ]:
with global_context as ctx:
    ctx[BATCH_SIZE_KEY] = 1
    _ = module(data)

print_graph_spec(module)

In [ ]:
# Let's record bs = 2 to have dynamic batch dimension

In [ ]:
with global_context as ctx:
    ctx[BATCH_SIZE_KEY] = 2
    _ = module(data.repeat(2, 1, 1, 1))

In [ ]:
print_graph_spec(module)

#### Let's try tune dry-run

In [ ]:
torch_compile_strategy = OneBackendStrategy(TorchInductorBackend())

In [ ]:
module.tune(strategy=torch_compile_strategy, dry_run=True, device=device)

#### Let's try real tune with torch compile

In [ ]:
module.tune(strategy=torch_compile_strategy, dry_run=False, device=device)

In [ ]:
out = module(data)
actual_probs = torch.nn.functional.softmax(out[0], dim=0)

In [ ]:
module.state

In [ ]:
torch.testing.assert_close(ref_probs, actual_probs, rtol=1e-4, atol=1e-4)